# 01 — WorldPop + H3 population exploration

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/01_worldpop_h3.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/01_worldpop_h3.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `3e2b39f2e8dfcc9cfcd791f78decae8146a0109b54a76bfd754b28d854c843f2`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 01 — WorldPop + H3 population exploration

> **Required runtime:** this notebook uses `geolibre_lite.LiteMap`, not upstream `geolibre.Map`.


## Start here: the question behind the code

**How does changing spatial scale change what we can compare?**

**What you will learn:** locate → outline → compare → join later. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Spatial indexing foundation: H3 assigns cells; it does not estimate their populations.

**Why this is useful:** A common spatial index can connect population with later model features while keeping scale choices visible.

**Where the work happens:** H3 geometry and area calculations run in Pyodide. The population PMTiles layer is streamed by the map viewer.

![The hexagon is a container. Assign one location to a resolution-7 H3 cell. Convert the cell boundary to longitude-first GeoJSON. Finer resolution covers a smaller area; geometry is independent of population. Use population attributes as a separate source before calculating exposure or density.](assets/learning/01/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: The hexagon is a container — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/01/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/01/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/01/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=01_worldpop_h3.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/01_worldpop_h3.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/01/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Locate (0–8 s):** Assign one location to a resolution-7 H3 cell. Code landmark: `latlng_to_cell(lat, lon, 7)`.

2. **Outline (8–16 s):** Convert the cell boundary to longitude-first GeoJSON. Code landmark: `cell_to_boundary(cell)`.

3. **Compare (16–24 s):** Finer resolution covers a smaller area; geometry is independent of population. Code landmark: `h3.cell_area(...)`.

4. **Join later (24–32 s):** Use population attributes as a separate source before calculating exposure or density. Code landmark: `m.add_pmtiles(...)`.


## See the connections

![Population and the scale of a hexagon: Population: WorldPop estimates people across space; Index: H3 groups locations at nested resolutions; Compare: Neighborhood map plus cell-area graph; Check: A hexagon is geometry, not a population count](assets/01_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("geolibre==3.0.0")
from geolibre_lite import LiteMap as Map
assert Map.__module__ == "geolibre_lite", f"Wrong Map loaded: {Map.__module__}"
print("Map implementation:", Map.__module__)


### Code guide 2 — Open a population tile archive

**Read the Python:** An uppercase variable stores the PMTiles URL. `add_pmtiles` lets the viewer request tiles for the current view. Python has not loaded a table of population totals here; it has configured a map layer.

**Look for:** A WorldPop H3 layer over Cambodia when the remote archive loads.

**Pause and try:** Zoom into Phnom Penh and inspect the legend or feature attributes before interpreting a color as a count.


In [ ]:
WORLDPOP_H3 = "https://data.source.coop/smartmaps/h3ys-worldpop/khm.pmtiles"

m = Map(center=(104.92, 12.55), zoom=6, height="700px")
m.add_pmtiles(WORLDPOP_H3, name="WorldPop population — H3 PMTiles")
m


## Create a real H3 cell in the browser

Current Pyodide distributions include the `h3` package, so we can use Uber's H3 indexing without a server.


### Code guide 3 — Turn one location into a hexagon

**Read the Python:** `h3.latlng_to_cell(lat, lon, 7)` assigns a spatial index at resolution 7. H3 accepts latitude first, whereas GeoJSON needs longitude first. The list comprehension swaps each boundary pair, then repeats the first point to close the polygon.

**Look for:** An H3 identifier in `cell` and a GeoJSON polygon in `feature`; neither contains a population count.

**Pause and try:** Change resolution 7 to 8. Predict whether the cell grows or shrinks.


In [ ]:
import h3

lat, lon = 11.5564, 104.9282  # Phnom Penh
cell = h3.latlng_to_cell(lat, lon, 7)
boundary = h3.cell_to_boundary(cell)

feature = {
    "type": "Feature",
    "properties": {"h3": cell, "resolution": 7, "place": "Phnom Penh"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[[lng, lat] for lat, lng in boundary] + [[boundary[0][1], boundary[0][0]]]],
    },
}
cell


### Code guide 4 — Overlay the hexagon and its reference point

**Read the Python:** A FeatureCollection wraps a list of features. `fillOpacity=0.25` leaves the population tiles visible underneath. `add_marker(lon, lat)` places the reference point using map coordinate order.

**Look for:** A highlighted cell around the Phnom Penh marker.

**Pause and try:** Switch the WorldPop layer off. What information remains, and what information disappears?


In [ ]:
m.add_geojson(
    {"type": "FeatureCollection", "features": [feature]},
    name="Example H3 r7 cell",
    fillColor="#ffcc00",
    fillOpacity=0.25,
    strokeColor="#111111",
    strokeWidth=2,
)
m.add_marker(lon, lat, name="Phnom Penh", properties={"h3": cell})
m


## Ideas to extend

- Use `h3.grid_disk(cell, k)` to build neighborhoods.
- Join indicators by H3 index instead of doing repeated polygon overlays.
- Compare population totals at multiple H3 resolutions to demonstrate scale effects.
- Use GeoLibre's PMTiles styling tools to create a graduated population map.

## Data & software citations

- WorldPop 2020 population-count product family: Bondarenko, M., Kerr, D., Sorichetta, A., & Tatem, A.J. (2020), WorldPop, University of Southampton. https://doi.org/10.5258/SOTON/WP00684
- WorldPop-on-H3 PMTiles mirror: UN Smart Maps Group, Source Cooperative, https://source.coop/smartmaps/h3ys-worldpop/khm.pmtiles
- H3: https://h3geo.org/ and https://github.com/uber/h3-py (Apache-2.0).
- GeoLibre: https://geolibre.app/.


### Code guide 5 — Load the plotting tools

**Read the Python:** `from IPython.display import display` lets a cell show objects explicitly. In Pyodide, `await loadPackage('matplotlib')` loads a compiled plotting package. The `if` block is skipped in ordinary Python, where the package must already be installed.

**Look for:** This preparation cell may show no figure; the next cell creates it.

**Pause and try:** Keep this cell before the chart cell when you rerun the notebook after a kernel restart.


In [ ]:
from IPython.display import display
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage("matplotlib")

## Compare neighborhood shape and spatial scale

The left panel is actual H3 geometry near Phnom Penh. The right panel measures the cell containing the same point at each resolution. Neither panel contains population estimates; the WorldPop PMTiles layer above supplies that separate variable.

### Code guide 6 — Measure the effect of spatial resolution

**Read the Python:** `grid_disk(cell, 2)` returns nearby H3 indexes; each becomes a polygon patch. The list comprehension calculates cell area in square kilometres for resolutions 3 through 9. `semilogy` uses a logarithmic vertical axis so very different areas fit together.

**Look for:** A neighborhood sketch and a falling cell-area curve. These are geometry measurements.

**Pause and try:** Try radius 3 and compare the number of cells. Explain why more cells alone does not mean more people.


In [ ]:
from cognition import style_plots
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
style_plots()
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for neighbor in h3.grid_disk(cell, 2):
    boundary_xy = [(lng, lat) for lat, lng in h3.cell_to_boundary(neighbor)]
    axes[0].add_patch(Polygon(boundary_xy, facecolor="#d97706" if neighbor == cell else "#dbeafe", edgecolor="white"))
axes[0].autoscale_view()
axes[0].set_aspect(1 / __import__('math').cos(__import__('math').radians(lat)))
axes[0].set(xlabel="Longitude", ylabel="Latitude", title="H3 r7: two rings around the center")
resolutions = list(range(3, 10))
areas = [h3.cell_area(h3.latlng_to_cell(lat, lon, r), unit="km^2") for r in resolutions]
axes[1].semilogy(resolutions, areas, "o-")
axes[1].set(xlabel="H3 resolution", ylabel="Cell area (km², log scale)", title="Finer cells cover less area")
plt.show()

**Predict → change → explain:** change `grid_disk(cell, 2)` to a radius of 3. Then compare r6 and r8. A sum of people can be aggregated; averaging cell counts without their areas does not produce population density.

## Make sense of the result

![More hexagons means more people? No: these are containers. Counts come from WorldPop. I will check area and population separately.](assets/learning/01/cartoon.svg)

**Cartoon transcript:** More hexagons means more people? → No: these are containers. Counts come from WorldPop. → I will check area and population separately.

**Explain it to a colleague:** If the same population is redistributed across finer cells, must the total grow?

<details><summary>Check your reasoning</summary>

No. Spatial resolution changes the allocation units; a correctly conserved sum can stay the same.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
